## Partie 1 : Exploration du dataset
On écrit une fonction `get_image_info` qui extrait, pour chaque image : nom, classe, format, mode, largeur, hauteur, écart-type des pixels, nombre de canaux et taille du fichier.
Les fichiers corrompus sont pris en charge avec un `try/except` : on garde leur ligne avec `corrupted=True` et les autres valeurs vides.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

In [4]:
def get_image_info(path: Path, label: str) -> dict:
    # Dictionnaire qui va contenir toutes les informations de l'image
    info = {
        # Nom du fichier
        "name": path.name,

        # Classe de l'image (ex: glass, plastic, metal...)
        "class": label,

        # Chemin complet de l'image
        "path": str(path),

        # Informations qui seront récupérées après ouverture de l'image
        "format": None,
        "mode": None,
        "width": None,
        "height": None,
        "std": None,
        "channels": None,

        # Taille du fichier en Ko
        "size_kb": round(path.stat().st_size / 1024, 2),

        # On considère l'image comme valide au départ
        "corrupted": False,
    }

    try:
        # Ouvre l'image avec Pillow
        with Image.open(path) as img:

            # Force le chargement complet de l'image
            # Cela permet notamment de détecter les fichiers tronqués ou corrompus
            img.load()

            # Convertit l'image en tableau NumPy
            arr = np.array(img)

            # Récupération des caractéristiques de l'image
            info.update({
                # Format du fichier : JPEG, PNG, WEBP...
                "format": img.format,

                # Mode de l'image : RGB, L, RGBA...
                "mode": img.mode,

                # Largeur de l'image en pixels
                "width": img.width,

                # Hauteur de l'image en pixels
                "height": img.height,

                # Écart-type des valeurs des pixels
                "std": float(arr.std()),

                # Nombre de canaux :
                # 1 si l'image est en niveaux de gris (2 dimensions)
                # sinon on récupère le nombre de canaux dans la 3e dimension
                "channels": 1 if arr.ndim == 2 else arr.shape[2],
            })

    # Si une erreur se produit lors de l'ouverture ou du chargement
    # l'image est considérée comme corrompue
    except Exception:
        info["corrupted"] = True

    # Retourne toutes les informations de l'image
    return info

On définit le chemin du dossier `raw` et la liste des classes, puis on parcourt chaque sous-dossier pour appliquer `get_image_info` à chaque fichier. Le résultat est rassemblé dans un DataFrame `df`.

In [5]:
RAW_DIR = Path("../data/raw")
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

rows = []
for label in CLASSES:
    for path in sorted((RAW_DIR / label).glob("*")):
        if path.is_file():
            rows.append(get_image_info(path, label))

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(1032, 11)


,name,class,path,format,mode,width,height,std,channels,size_kb,corrupted
0,cardboard1.jpg,cardboard,..\data\raw\cardboard\cardboard1.jpg,JPEG,RGB,512.0,384.0,40.588529,3.0,16.93,False
1,cardboard10.jpg,cardboard,..\data\raw\cardboard\cardboard10.jpg,JPEG,RGB,512.0,384.0,42.571288,3.0,21.17,False
2,cardboard100.jpg,cardboard,..\data\raw\cardboard\cardboard100.jpg,JPEG,RGB,512.0,384.0,46.108305,3.0,14.54,False
3,cardboard101.jpg,cardboard,..\data\raw\cardboard\cardboard101.jpg,JPEG,RGB,512.0,384.0,72.263996,3.0,13.95,False
4,cardboard102.jpg,cardboard,..\data\raw\cardboard\cardboard102.jpg,JPEG,RGB,512.0,384.0,48.388937,3.0,17.59,False


On vérifie la structure du DataFrame : nombre de lignes, types des colonnes et valeurs manquantes. Les valeurs manquantes correspondent aux fichiers corrompus.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1032 entries, 0 to 1031
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       1032 non-null   str    
 1   class      1032 non-null   str    
 2   path       1032 non-null   str    
 3   format     1026 non-null   str    
 4   mode       1026 non-null   str    
 5   width      1026 non-null   float64
 6   height     1026 non-null   float64
 7   std        1026 non-null   float64
 8   channels   1026 non-null   float64
 9   size_kb    1032 non-null   float64
 10  corrupted  1032 non-null   bool   
dtypes: bool(1), float64(5), str(5)
memory usage: 81.8 KB


## Partie 2 : Détecter les images corrompues
On écrit une fonction `is_corrupted(path)` qui renvoie `True` si l'image ne peut pas être ouverte ou décodée entièrement.
On procède en deux temps : `verify()` contrôle la structure du fichier, puis on rouvre l'image et on appelle `load()` pour forcer le décodage complet des pixels (`verify()` seul ne détecte pas tous les fichiers tronqués).

In [7]:
def is_corrupted(path: Path) -> bool:
    try:
        with Image.open(path) as img:
            img.verify()      # vérifie la structure du fichier
        with Image.open(path) as img:
            img.load()        # force le décodage complet des pixels
        return False
    except Exception:
        return True

On applique la fonction à toutes les images du DataFrame, puis on affiche les images corrompues.

In [8]:
df["is_corrupted"] = df["path"].apply(lambda p: is_corrupted(Path(p)))

print("Nombre d'images corrompues :", df["is_corrupted"].sum())
df[df["is_corrupted"]][["name", "class", "size_kb"]]

Nombre d'images corrompues : 6


,name,class,size_kb
147,cardboard83.jpg,cardboard,20.40
326,glass74.jpg,glass,7.02
446,metal48.jpg,metal,7.09
633,paper213.jpg,paper,7.12
791,plastic13.jpg,plastic,5.13
1004,trash3.jpg,trash,8.98


## Partie 3 : Détecter les images vides
On écrit une fonction `is_empty(path, std_threshold)` qui renvoie `True` si l'image est :
- entièrement noire (tous les pixels valent 0) ;
- entièrement blanche (tous les pixels valent 255) ;
- quasi uniforme : l'écart-type des pixels est inférieur à un seuil, donc très peu de variation.

On convertit d'abord l'image en niveaux de gris (`L`) pour travailler sur un seul canal. Les images corrompues renvoient `False` ici, car elles sont déjà traitées à la Partie 2.

In [9]:
def is_empty(path: Path, std_threshold: float = 5.0) -> bool:
    try:
        with Image.open(path) as img:
            gray = np.array(img.convert("L"))
    except Exception:
        return False  # image corrompue : gérée à la Partie 2

    if gray.max() == 0:        # entièrement noire
        return True
    if gray.min() == 255:      # entièrement blanche
        return True
    return gray.std() < std_threshold  # très peu de variation

On applique la fonction à toutes les images, puis on affiche celles détectées comme vides, triées par écart-type croissant.

In [10]:
df["is_empty"] = df["path"].apply(lambda p: is_empty(Path(p)))

print("Nombre d'images vides ou quasi vides :", df["is_empty"].sum())
df[df["is_empty"]][["name", "class", "std"]].sort_values("std")

Nombre d'images vides ou quasi vides : 4


,name,class,std
167,image-blanche-512x384.jpg,cardboard,1.572536
357,image-blanche-512x384.jpg,metal,1.572536
355,image-noire-512x384.png,glass,110.418239
358,image-noire-512x384.png,metal,110.418239


## Partie 4 : Détecter les différences de résolution
On isole les images lisibles (non corrompues), on convertit largeur et hauteur en entiers (elles étaient en décimaux à cause des valeurs vides), puis on crée deux colonnes : `resolution` (au format `largeurxhauteur`) et `pixels` (largeur × hauteur, pour comparer la taille des images).

In [13]:
df_valid = df[~df["corrupted"]].copy()

df_valid["width"] = df_valid["width"].astype(int)
df_valid["height"] = df_valid["height"].astype(int)
df_valid["resolution"] = df_valid["width"].astype(str) + "x" + df_valid["height"].astype(str)
df_valid["pixels"] = df_valid["width"] * df_valid["height"]

df_valid[["name", "width", "height", "resolution", "pixels"]].head()

,name,width,height,resolution,pixels
0,cardboard1.jpg,512,384,512x384,196608
1,cardboard10.jpg,512,384,512x384,196608
2,cardboard100.jpg,512,384,512x384,196608
3,cardboard101.jpg,512,384,512x384,196608
4,cardboard102.jpg,512,384,512x384,196608


### 4.1 Résolution minimale, maximale, résolutions les plus fréquentes et nombre d'images par résolution
La résolution minimale et maximale sont celles de l'image qui a le moins et le plus de pixels au total.

In [14]:
i_min = df_valid["pixels"].idxmin()
i_max = df_valid["pixels"].idxmax()

print("Résolution minimale :", df_valid.loc[i_min, "resolution"], "->", df_valid.loc[i_min, "name"])
print("Résolution maximale :", df_valid.loc[i_max, "resolution"], "->", df_valid.loc[i_max, "name"])

Résolution minimale : 32x32 -> cardboard22.jpg
Résolution maximale : 512x384 -> cardboard1.jpg


In [15]:
res_counts = df_valid["resolution"].value_counts()
print("Nombre de résolutions différentes :", len(res_counts))
res_counts.head(10)

Nombre de résolutions différentes : 4


resolution
512x384    1013
32x32         5
48x32         4
40x40         4
Name: count, dtype: int64

### 4.2 Images plus petites que 64 × 64
Une image respecte la contrainte si sa largeur **et** sa hauteur sont au moins égales à `MIN_SIZE`. On ajoute une colonne `too_small` au DataFrame principal pour la réutiliser dans l'audit.

In [16]:
MIN_SIZE = 64

df["too_small"] = (df["width"] < MIN_SIZE) | (df["height"] < MIN_SIZE)

print("Nombre d'images trop petites :", df["too_small"].sum())
df[df["too_small"]][["name", "class", "width", "height"]]

Nombre d'images trop petites : 13


,name,class,width,height
20,cardboard117.jpg,cardboard,48.0,32.0
77,cardboard22.jpg,cardboard,32.0,32.0
133,cardboard70.jpg,cardboard,40.0,40.0
171,glass100.jpg,glass,40.0,40.0
229,glass15.jpg,glass,48.0,32.0
268,glass21.jpg,glass,32.0,32.0
270,glass23.jpg,glass,32.0,32.0
383,metal121.jpg,metal,48.0,32.0
413,metal2.jpg,metal,32.0,32.0
421,metal26.jpg,metal,40.0,40.0


## Partie 5 : Détecter les différents canaux
Le nombre de canaux indique le type de l'image :
- 1 canal : niveaux de gris (mode `L`)
- 3 canaux : couleur RGB
- 4 canaux : RGB + transparence, RGBA (mode `RGBA`)

On compte le nombre d'images pour chaque valeur de `channels`. La colonne était en décimaux à cause des valeurs vides, on la convertit en entiers.

In [17]:
df_valid["channels"] = df_valid["channels"].astype(int)

channel_counts = df_valid["channels"].value_counts().sort_index()
channel_counts

channels
1       2
3    1006
4      18
Name: count, dtype: int64

On croise avec le mode Pillow (`L`, `RGB`, `RGBA`, `P`...) pour vérifier que les canaux correspondent bien aux modes attendus.

In [18]:
pd.crosstab(df_valid["mode"], df_valid["channels"])

channels,1,3,4
mode,,,
P,2,0,0
RGB,0,1006,0
RGBA,0,0,18


## Partie 6 : Détecter les doublons
On écrit une fonction `image_hash(path)` qui calcule une empreinte MD5 à partir du contenu des pixels : l'image est convertie en RGB, puis on hache ses dimensions et ses octets. Deux images de même contenu ont la même empreinte, même avec des noms de fichier différents. Les images corrompues renvoient `None`.

In [21]:
import hashlib

def image_hash(path: Path):
    # On essaie d'ouvrir et de lire l'image
    try:
        with Image.open(path) as img:

            # On convertit l'image en RGB
            # pour avoir le même format pour toutes les images
            rgb = img.convert("RGB")

            # On crée un hash MD5 vide
            h = hashlib.md5()

            # On ajoute les dimensions de l'image au hash
            # Exemple : (512, 384)
            h.update(str(rgb.size).encode())

            # On ajoute les données des pixels au hash
            h.update(rgb.tobytes())

            # On retourne l'empreinte MD5 sous forme de texte
            return h.hexdigest()

    # Si l'image est corrompue ou impossible à lire
    except Exception:
        # On retourne None
        return None


On calcule l'empreinte de chaque image, puis on marque comme doublon toute image dont l'empreinte est déjà apparue plus haut dans le tableau (`keep="first"` conserve la première occurrence). Les images corrompues (empreinte vide) sont exclues du calcul.

In [22]:
df["hash"] = df["path"].apply(lambda p: image_hash(Path(p)))

df["is_duplicate"] = df["hash"].notna() & df.duplicated(subset="hash", keep="first")

print("Nombre de doublons :", df["is_duplicate"].sum())

Nombre de doublons : 15


On affiche les groupes d'images identiques (toutes les copies, y compris la première), triés par empreinte, pour voir quels fichiers et quelles classes sont concernés.

In [23]:
dup_groups = df[df["hash"].notna() & df.duplicated(subset="hash", keep=False)]
dup_groups.sort_values("hash")[["name", "class", "hash"]]

,name,class,hash
757,paperer35.jpg,paper,084d6885beb9cd78023af968673c277a
686,paper35.jpg,paper,084d6885beb9cd78023af968673c277a
804,plastic140y.jpg,plastic,19b5274a449d2eeecb64f8043b7f1c0a
803,plastic140.jpg,plastic,19b5274a449d2eeecb64f8043b7f1c0a
167,image-blanche-512x384.jpg,cardboard,4506e56263533f8f8c509b66ab10dd82
357,image-blanche-512x384.jpg,metal,4506e56263533f8f8c509b66ab10dd82
732,paper77ty.jpg,paper,53bdf6ad66903e0fc02b35b3cad77a43
731,paper77.jpg,paper,53bdf6ad66903e0fc02b35b3cad77a43
910,plastic36rt.jpg,plastic,6945d4f24986bcb2a77cdfca092496b1
909,plastic36.jpg,plastic,6945d4f24986bcb2a77cdfca092496b1


## Partie 7 : Détecter les images mal classées

Le contrôle a été fait **visuellement et à la main** : chaque dossier de `data/raw` a été parcouru dans l'explorateur de fichiers en affichage miniatures, et les images qui ne correspondent pas à leur classe ont été relevées.

### Résultat du contrôle visuel

| N° | Image | Classe du dossier | Classe réelle | Ce que montre l'image |
|----|-------|-------------------|---------------|-----------------------|
| 1 | cardboard86d | cardboard | plastic | une bouteille |
| 2 | cardboard161 | cardboard | glass | un objet en verre |
| 3 | glass12er | glass | paper | du papier |
| 4 | metal150 | metal | cardboard | un carton |
| 5 | paper30 | paper | cardboard | un carton |
| 6 | paper200 | paper | cardboard | un carton |
| 7 | paper201 | paper | metal | un objet en métal |
| 8 | plastic109 | plastic | cardboard | un carton |

### Conclusion de la Partie 7

- **8 images mal classées** ont été détectées.
- La classe `cardboard` est la classe réelle de 4 d'entre elles, mais elle contient aussi 2 intruses (une bouteille en plastique et un objet en verre).
- Ces erreurs sont dangereuses pour l'entraînement : le modèle apprendrait des associations fausses (par exemple « bouteille = carton »).
- Le dossier `raw` reste **inchangé** (lecture seule). Ces 8 images seront traitées lors de la construction du dossier `cleaned`.
- Les images `image-blanche-512x384.jpg` et `image-noire-512x384.png`, présentes dans plusieurs classes, sont comptées à part avec les images vides (Partie 3) et les doublons (Partie 6).

## Partie 8 : Analyser le déséquilibre des classes
On compte le nombre d'images par classe dans le dataset brut, puis on calcule leur pourcentage. Une classe est dite **minoritaire** quand elle a nettement moins d'images que les autres : le modèle a alors moins d'exemples pour l'apprendre et a tendance à la négliger.

In [25]:
class_counts = df["class"].value_counts()

class_stats = pd.DataFrame({
    "nb_images": class_counts,
    "pourcentage": (class_counts / len(df) * 100).round(1),
})
class_stats

,nb_images,pourcentage
class,,
paper,252,24.4
plastic,224,21.7
glass,188,18.2
cardboard,169,16.4
metal,149,14.4
trash,50,4.8


**Observations :**
- Le dataset est **déséquilibré**. Avec 6 classes, une répartition parfaitement équilibrée donnerait environ 16,7 % par classe.
- La classe majoritaire est `paper` (252 images) et la classe minoritaire est `trash` (50 images) : le rapport est d'environ **5 pour 1**.
- `trash` est très en dessous des autres classes (4,8 % contre au moins 14,4 % pour les autres).
- Ce comptage porte sur le dataset **brut** : il inclut encore les images corrompues, vides, dupliquées et mal classées. Il sera recalculé sur le dataset nettoyé.

**Risque :** un modèle entraîné sur ces données verra très peu d'exemples de `trash` et aura tendance à l'ignorer, même s'il obtient une bonne précision globale.

**Conséquence :** la classe `trash` recevra la data augmentation, après le découpage train/validation/test.